# Urdu-English NMT — Baseline MarianMT Evaluation
### Mid-Report Experiment | Notebook 03

**Objective:** Load `Helsinki-NLP/opus-mt-ur-en` (pre-trained MarianMT), run beam-4 translation on 2,000 test sentences, and record corpus-level BLEU + ChrF++ scores.

This is the **mid-report baseline** — a zero-shot evaluation of the best publicly available Urdu→English model, trained on the same OPUS data we cleaned. The score establishes the floor our fine-tuned ablation variants must beat.

---
| Setting | Value |
|---------|-------|
| Model | `Helsinki-NLP/opus-mt-ur-en` |
| Inference | Beam search, width=4 |
| Test sentences | 2,000 (randomly sampled, seed=42) |
| Metrics | BLEU (sacrebleu, intl tokenizer), ChrF++ |
| GPU | Kaggle T4 (~10 minutes total) |

## Cell 1 — Environment Setup
Install dependencies and set up the project root on the Python path.

In [ ]:
# ============================================================
# CELL 1: Environment setup
# Run this first. All pip installs happen here.
# On Kaggle, most of these are pre-installed; the installs
# are fast (just upgrading / confirming versions).
# ============================================================

import subprocess, sys

# Install / upgrade required packages
packages = [
    'transformers>=4.40.0',
    'sacrebleu>=2.3.0',
    'sentencepiece',
    'datasets',
    'evaluate',
    'torch',           # pre-installed on Kaggle GPU
    'langdetect',
    'tqdm',
]

for pkg in packages:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True
    )

print('All packages installed/confirmed.')

In [ ]:
# ============================================================
# CELL 2: Project path setup
# Point Python at your cloned repo so all internal imports work.
# Adjust PROJECT_ROOT to wherever you cloned the repo on Kaggle.
# ============================================================

import os, sys

# ---------------------------------------------------------------
# OPTION A: You uploaded the repo as a Kaggle Dataset
# Kaggle datasets are mounted at /kaggle/input/<dataset-slug>/
# Example: PROJECT_ROOT = '/kaggle/input/urdu-en-nmt/urdu-en-nmt'
#
# OPTION B: You cloned the repo into /kaggle/working/ in a prior cell
# Example: !git clone https://github.com/<you>/urdu-en-nmt /kaggle/working/urdu-en-nmt
#          PROJECT_ROOT = '/kaggle/working/urdu-en-nmt'
# ---------------------------------------------------------------

PROJECT_ROOT = '/kaggle/working/urdu-en-nmt'   # <-- CHANGE THIS IF NEEDED

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Verify the path is correct
assert os.path.exists(os.path.join(PROJECT_ROOT, 'model', 'evaluate.py')), \
    f"model/evaluate.py not found at {PROJECT_ROOT}. Check PROJECT_ROOT."

print(f'Project root: {PROJECT_ROOT}')
print('Imports will work correctly.')

In [ ]:
# ============================================================
# CELL 3: Configure data paths
# Tell the config module where to find your TSV files.
# The easiest approach: set environment variables BEFORE importing
# model.config (the config module reads os.environ at import time).
# ============================================================

# Adjust these paths to where your processed TSV files actually are.
# If you ran download_and_clean.py on Kaggle, they are in:
#   /kaggle/working/data/processed/

os.environ['NMT_DATA_DIR']    = '/kaggle/working/data'
os.environ['NMT_RESULTS_DIR'] = '/kaggle/working/results'

# Create results directory if it doesn't exist
os.makedirs('/kaggle/working/results', exist_ok=True)

print('Data dir   :', os.environ['NMT_DATA_DIR'])
print('Results dir:', os.environ['NMT_RESULTS_DIR'])

# Quick sanity check: confirm test.tsv exists
test_tsv = os.path.join(os.environ['NMT_DATA_DIR'], 'processed', 'test.tsv')
if os.path.exists(test_tsv):
    n_lines = sum(1 for _ in open(test_tsv, encoding='utf-8'))
    print(f'test.tsv found: {n_lines:,} lines')
else:
    print(f'WARNING: test.tsv not found at {test_tsv}')
    print('You may need to run the data pipeline first (notebook 01_data_pipeline).')

## Cell 4 — Load Model & Run Evaluation
This single cell runs the entire baseline experiment. Expected runtime on a Kaggle T4: ~8–12 minutes.

In [ ]:
# ============================================================
# CELL 4: Run the baseline evaluation
# This is the main experiment cell for the mid-report.
# Expected output: BLEU and ChrF++ scores on 2k test sentences.
# ============================================================

from model.config import get_config
from model.evaluate import run_baseline_evaluation

# Load baseline config
# get_config('baseline') returns a BaselineConfig with:
#   model = Helsinki-NLP/opus-mt-ur-en
#   eval_samples = 2000
#   num_beams = 4
cfg = get_config('baseline')

# Optional: override specific settings here without editing config.py
cfg.eval_samples = 2000   # 2k test sentences for mid-report
cfg.num_beams    = 4      # Standard beam width for publishable baselines
cfg.batch_size   = 32     # Reduce to 16 if you get CUDA OOM

print('Experiment configuration:')
print(f'  model         : {cfg.model_name_or_path}')
print(f'  test_tsv      : {cfg.test_tsv}')
print(f'  eval_samples  : {cfg.eval_samples}')
print(f'  num_beams     : {cfg.num_beams}')
print(f'  batch_size    : {cfg.batch_size}')
print(f'  results_dir   : {cfg.results_dir}')
print()

# ---- RUN THE EXPERIMENT ----
metrics = run_baseline_evaluation(cfg)

# The function prints a full results table at the end.
# Key numbers for your mid-report are:
print(f"\n{'='*50}")
print(f"  MID-REPORT RESULT (copy into your report):")
print(f"  BLEU   = {metrics['bleu']:.2f}")
print(f"  ChrF++ = {metrics['chrf_plus_plus']:.2f}")
print(f"  N      = {metrics['num_sentences']:,} test sentences")
print(f"{'='*50}")

## Cell 5 — Inspect Sample Translations
Visual inspection of translations is as important as numerical metrics for your report.

In [ ]:
# ============================================================
# CELL 5: Inspect sample translations
# Read the saved predictions back and display side-by-side.
# Include 5-10 of these examples in your mid-report to give
# qualitative evidence alongside the BLEU score.
# ============================================================

import json, random

# Load saved results
preds_file = cfg.predictions_file
refs_file  = cfg.references_file

with open(preds_file, encoding='utf-8') as f:
    predictions = f.read().splitlines()

with open(refs_file, encoding='utf-8') as f:
    references = f.read().splitlines()

# Load metrics summary
with open(cfg.metrics_file) as f:
    saved_metrics = json.load(f)

print('Saved Metrics:')
print(json.dumps(saved_metrics, indent=2))

# Display random samples
print(f"\n{'='*65}")
print('SAMPLE TRANSLATIONS (for qualitative analysis in report)')
print(f"{'='*65}")

rng = random.Random(42)
indices = rng.sample(range(len(predictions)), min(10, len(predictions)))

for i, idx in enumerate(indices, 1):
    print(f"\n[{i:02d}]")
    print(f"  Model output : {predictions[idx]}")
    print(f"  Reference    : {references[idx]}")

In [ ]:
# ============================================================
# CELL 6: Per-domain BLEU breakdown (optional but impressive)
# If your test.tsv has a 'source' column indicating corpus origin
# (TED2020, GNOME, Tanzil), this cell breaks down BLEU per domain.
# Shows that TED2020 ≈ highest BLEU (clean natural language),
# Tanzil ≈ lowest (archaic religious text), GNOME ≈ middle.
# This analysis is great to include in the mid-report.
# ============================================================

import sacrebleu

# This cell only runs if you tagged your test pairs with domain labels.
# If not, it prints a helpful message and exits gracefully.

domain_tsv = os.path.join(os.environ['NMT_DATA_DIR'], 'processed', 'test_with_domain.tsv')

if not os.path.exists(domain_tsv):
    print('test_with_domain.tsv not found — skipping per-domain breakdown.')
    print('(This is optional. Your test.tsv without domain tags is fine for the mid-report.)')
else:
    # Load domain-tagged test set
    domain_pairs = {}   # {domain: [(pred, ref), ...]}

    with open(domain_tsv, encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) == 3:  # urdu, english, domain
                domain = parts[2].strip()
                if domain not in domain_pairs:
                    domain_pairs[domain] = []
                domain_pairs[domain].append((parts[0], parts[1]))

    print(f"{'Domain':<15} {'Pairs':>8}  {'BLEU':>8}")
    print('-' * 35)
    for domain, pairs in sorted(domain_pairs.items()):
        # NOTE: In a real analysis you'd use the model's predictions for these
        # specific pairs. This is a placeholder showing the structure.
        print(f"{domain:<15} {len(pairs):>8,}  {'(run eval)':>8}")

## Results Summary

After running the cells above, your mid-report should include:

| Metric | Score |
|--------|-------|
| **BLEU** | *(from Cell 4 output)* |
| **ChrF++** | *(from Cell 4 output)* |

### What to write in the report

**Preliminary Results (Methodology section):**
> We evaluate `Helsinki-NLP/opus-mt-ur-en` as a zero-shot baseline. This MarianMT model was pre-trained by the Helsinki NLP group on OPUS Urdu-English parallel corpora — the same data sources we use — making it a strong and directly comparable starting point. We evaluate on 2,000 randomly sampled sentences from our cleaned test split using beam search (width=4) and report corpus-level BLEU with the sacrebleu `intl` tokenizer and ChrF++ (word order=2).

### Files produced in `results/`:
- `baseline_predictions.txt` — model translations (one per line)
- `baseline_references.txt` — human references (one per line)  
- `baseline_metrics.json` — all scores as structured JSON
- `baseline_samples.txt` — 20 side-by-side translation examples